# Claude Code-style LangGraph Harness

这套实现把 Claude Code 的核心阶段显式写成 LangGraph：

```text
prepare → plan(TodoWrite) → model ↔ tools → compact → finalize
```

重点不是“调用 Claude 模型”，而是复刻 Harness：

- `CLAUDE.md` 分层上下文；
- `queryLoop` 的 tool_use/tool_result 循环；
- permission modes；
- microcompact/full compact；
- session memory；
- `Task` / `SendMessage` / `TaskOutput` 子 Agent；
- MCP 动态工具；
- 事件 JSONL 和 timeline。

详细说明：`docs/claude-harness-architecture.md`


In [ ]:
from pathlib import Path
import shutil, tempfile
from claude_harness import ClaudeHarnessConfig, ClaudeHarness
from claude_harness.context import discover_claude_md

source = Path("examples/claude_harness_demo").resolve()
workspace = Path(tempfile.mkdtemp(prefix="claude_harness_nb_")) / "demo"
shutil.copytree(source, workspace)
print("workspace:", workspace)
print("CLAUDE.md:\n", discover_claude_md(workspace))


## 1. 工具与权限

Claude Code 风格工具名：`Read`、`Glob`、`Grep`、`Explore`、`Write`、`Edit`、`Bash`、`TodoWrite`。
本项目另外加载 `Task`、`SendMessage`、`TaskOutput`、`TaskList`、`CloseAgent`，以及 MCP 的只读工具。

In [ ]:
from claude_harness.events import EventBus
from claude_harness.hooks import HookManager
from claude_harness.permissions import PermissionManager
from claude_harness.state import ChangeTracker, TodoState
from claude_harness.tools import ToolContext, build_tools

bus = EventBus(workspace / ".claude" / "runs" / "tool-preview", console=False)
ctx = ToolContext(workspace, TodoState(), ChangeTracker(), [], PermissionManager("acceptEdits"), HookManager(), bus)
tools, meta = build_tools(ctx)
for tool in tools:
    print(f"{tool.name:12s} read_only={meta[tool.name].read_only}  {tool.description[:45]}")
bus.close()


## 2. 运行完整开发任务

下面的任务会要求模型：

1. 用 `Task` 派发只读 Explore 子 Agent；
2. 用 `TaskOutput` 等待结果；
3. 调用 MCP 的 `repo_stats`；
4. 修改 `calculator.py`；
5. 运行 pytest；
6. 更新 Todo。

这会调用真实模型，耗时约 30–90 秒。

In [ ]:
config = ClaudeHarnessConfig(
    workspace=workspace,
    permission_mode="acceptEdits",
    auto_approve=True,
    enable_mcp=True,
    verify_command="python -m pytest -q",
    max_turns=12,
    console=True,
)
result = ClaudeHarness(config).run(
    "请先调用 Task 启动 explore 子 Agent，并调用 TaskOutput 获取结果；"
    "再调用 MCP repo_stats；然后修复 calculator.py 中 add 的 bug，运行 pytest，并更新 Todo。"
)
print("success:", result.success)
print("changed_files:", result.changed_files)
print("compactions:", result.compactions)
print("run_dir:", result.run_dir)


## 3. 查看可视化产物

- `events.jsonl`：每个阶段/工具/子 Agent/MCP 事件；
- `timeline.md`：阶段耗时；
- `summary.md`：最终总结；
- `changes.diff`：文件变更。

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result.timeline_path.read_text(encoding="utf-8")))
display(Markdown((result.run_dir / "summary.md").read_text(encoding="utf-8")))
print((result.run_dir / "changes.diff").read_text(encoding="utf-8"))


In [ ]:
import json
events = [json.loads(line) for line in (result.run_dir / "events.jsonl").read_text(encoding="utf-8").splitlines()]
for event in events:
    print(f"{event['stage']:10s} {event['type']:28s} {event['message'].replace(chr(10), ' ')[:130]}")
